In [1]:
import os
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pandas.api.types import CategoricalDtype
from category_encoders import MEstimateEncoder
from sklearn.model_selection import KFold, cross_val_score
from sklearn.feature_selection import mutual_info_regression
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')

# Set styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rc('figure', autolayout=True)

# Step 1: Data Preprocessing & Cleaning

In [2]:
def clean_data(df):
    # Fix known typos in Kaggle Ames dataset
    df['Exterior2nd'] = df['Exterior2nd'].replace({'Brk Cmn': 'BrkComm'})
    
    # Fix corrupt GarageYrBlt values (replace with YearBuilt)
    df['GarageYrBlt'] = df['GarageYrBlt'].where(df['GarageYrBlt'] <= 2010, df['YearBuilt'])
    
    # Clean awkward column names starting with numbers
    df.rename(
        columns={
            '1stFlrSF': 'FirstFlrSF',
            '2ndFlrSF': 'SecondFlrSF',
            '3SsnPorch': 'Threeseasonporch',
        },
        inplace=True,
    )
    return df


# Define Nominal and Ordinal Categorical Features
features_nom = [
    'MSSubClass', 'MSZoning', 'Street', 'Alley', 'LandContour', 'LotConfig', 
    'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 
    'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 
    'Foundation', 'Heating', 'CentralAir', 'GarageType', 'MiscFeature', 
    'SaleType', 'SaleCondition'
]

five_levels = ['Po', 'Fa', 'TA', 'Gd', 'Ex']
ten_levels = list(range(10))

ordered_levels = {
    'OverallQual': ten_levels,
    'OverallCond': ten_levels,
    'ExterQual': five_levels,
    'ExterCond': five_levels,
    'BsmtQual': five_levels,
    'BsmtCond': five_levels,
    'HeatingQC': five_levels,
    'KitchenQual': five_levels,
    'FireplaceQu': five_levels,
    'GarageQual': five_levels,
    'GarageCond': five_levels,
    'PoolQC': five_levels,
    'LotShape': ['Reg', 'IR1', 'IR2', 'IR3'],
    'LandSlope': ['Sev', 'Mod', 'Gtl'],
    'BsmtExposure': ['No', 'Mn', 'Av', 'Gd'],
    'BsmtFinType1': ['Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'BsmtFinType2': ['Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'Functional': ['Sal', 'Sev', 'Maj1', 'Maj2', 'Mod', 'Min2', 'Min1', 'Typ'],
    'GarageFinish': ['Unf', 'RFn', 'Fin'],
    'PavedDrive': ['N', 'P', 'Y'],
    'Utilities': ['NoSeWa', 'NoSewr', 'AllPub'],
    'CentralAir': ['N', 'Y'],
    'Electrical': ['Mix', 'FuseP', 'FuseF', 'FuseA', 'SBrkr'],
    'Fence': ['MnWw', 'GdWo', 'MnPrv', 'GdPrv'],
}
ordered_levels = {key: ['None'] + value for key, value in ordered_levels.items()}


def encode_data(df):
    for name in features_nom:
        df[name] = df[name].astype('category')
        if 'None' not in df[name].cat.categories:
            df[name] = df[name].cat.add_categories('None')
            
    for name, levels in ordered_levels.items():
        if name in df.columns:
            df[name] = df[name].astype(CategoricalDtype(levels, ordered=True))
    return df


def impute_data(df):
    for name in df.select_dtypes('number').columns:
        df[name] = df[name].fillna(0)
    for name in df.select_dtypes('category').columns:
        df[name] = df[name].fillna('None')
    return df


def load_data(data_dir=Path('.')):
    df_train = pd.read_csv(data_dir / 'train.csv', index_col='Id')
    df_test = pd.read_csv(data_dir / 'test.csv', index_col='Id')
    
    # Combine train & test for consistent cleaning/encoding
    df = pd.concat([df_train, df_test], axis=0)
    df = clean_data(df)
    df = encode_data(df)
    df = impute_data(df)
    
    df_train = df.loc[df_train.index, :].copy()
    df_test = df.loc[df_test.index, :].copy()
    return df_train, df_test

# Step 2: Scoring & Baseline Evaluation Function

In [3]:
def score_dataset(X, y, model=XGBRegressor(random_state=42)):
    X_encoded = X.copy()
    for colname in X_encoded.select_dtypes(['category']).columns:
        X_encoded[colname] = X_encoded[colname].cat.codes
        
    log_y = np.log(y)
    score = cross_val_score(
        model, X_encoded, log_y, cv=5, scoring='neg_mean_squared_error'
    )
    rmsle = np.sqrt(-score.mean())
    return rmsle

# Step 3: Mutual Information (Dropping Uninformative Features)

In [4]:
def make_mi_scores(X, y):
    X_encoded = X.copy()
    for colname in X_encoded.select_dtypes(['object', 'category']).columns:
        X_encoded[colname], _ = X_encoded[colname].factorize()
    discrete_features = [pd.api.types.is_integer_dtype(t) for t in X_encoded.dtypes]
    mi_scores = mutual_info_regression(X_encoded, y, discrete_features=discrete_features, random_state=0)
    mi_scores = pd.Series(mi_scores, name='MI Scores', index=X_encoded.columns)
    return mi_scores.sort_values(ascending=False)

# Step 4: Feature Engineering Pipeline + Cross-Fold Target Encoding

In [5]:
# class CrossFoldEncoder:
#     def __init__(self, encoder, **kwargs):
#         self.encoder_ = encoder
#         self.kwargs_ = kwargs
#         self.cv_ = KFold(n_splits=5, shuffle=True, random_state=42)

#     def fit_transform(self, X, y, cols):
#         self.fitted_encoders_ = []
#         self.cols_ = cols
#         X_encoded = []
#         for idx_encode, idx_train in self.cv_.split(X, y):
#             fitted_encoder = self.encoder_(cols=cols, **self.kwargs_)
#             fitted_encoder.fit(X.iloc[idx_encode, :], y.iloc[idx_encode])
#             X_encoded.append(fitted_encoder.transform(X.iloc[idx_train, :])[cols])
#             self.fitted_encoders_.append(fitted_encoder)
#         X_encoded = pd.concat(X_encoded)
#         X_encoded.columns = [name + '_encoded' for name in X_encoded.columns]
#         return X_encoded

#     def transform(self, X):
#         from functools import reduce
#         X_encoded_list = []
#         for fitted_encoder in self.fitted_encoders_:
#             X_encoded = fitted_encoder.transform(X)
#             X_encoded_list.append(X_encoded[self.cols_])
#         X_encoded = reduce(lambda x, y: x.add(y, fill_value=0), X_encoded_list) / len(X_encoded_list)
#         X_encoded.columns = [name + '_encoded' for name in X_encoded.columns]
#         return X_encoded
class CrossFoldEncoder:

  def __init__(self, encoder, **kwargs):
    self.encoder_ = encoder
    self.kwargs_ = kwargs
    self.cv_ = KFold(n_splits=5, shuffle=True, random_state=42)

  def fit_transform(self, X, y, cols):
    self.fitted_encoders_ = []
    self.cols_ = cols
    X_encoded = []
    for idx_encode, idx_train in self.cv_.split(X, y):
      fitted_encoder = self.encoder_(cols=cols, **self.kwargs_)
      # Fit only on the target columns `cols`
      fitted_encoder.fit(X.iloc[idx_encode][cols], y.iloc[idx_encode])
      X_encoded.append(
          fitted_encoder.transform(X.iloc[idx_train][cols])[cols]
      )
      self.fitted_encoders_.append(fitted_encoder)
    X_encoded = pd.concat(X_encoded)
    X_encoded.columns = [name + "_encoded" for name in X_encoded.columns]
    return X_encoded

  def transform(self, X):
    from functools import reduce

    X_encoded_list = []
    for fitted_encoder in self.fitted_encoders_:
      # Transform only on `cols`
      X_encoded = fitted_encoder.transform(X[self.cols_])
      X_encoded_list.append(X_encoded[self.cols_])
    X_encoded = (
        reduce(lambda x, y: x.add(y, fill_value=0), X_encoded_list)
        / len(X_encoded_list)
    )
    X_encoded.columns = [name + "_encoded" for name in X_encoded.columns]
    return X_encoded


def mathematical_transforms(df):
    X = pd.DataFrame()
    X['LivLotRatio'] = df.GrLivArea / (df.LotArea + 1e-5)
    X['Spaciousness'] = (df.FirstFlrSF + df.SecondFlrSF) / (df.TotRmsAbvGrd + 1e-5)
    return X


def interactions(df):
    X = pd.get_dummies(df.BldgType, prefix='Bldg', drop_first=True)
    X = X.mul(df.GrLivArea, axis=0)
    return X


def counts(df):
    X = pd.DataFrame()
    X['PorchTypes'] = df[['WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', 'Threeseasonporch', 'ScreenPorch']].gt(0.0).sum(axis=1)
    return X


def group_transforms(df):
    X = pd.DataFrame()
    X['MedNhbdArea'] = df.groupby('Neighborhood', observed=False)['GrLivArea'].transform('median')
    return X


def pca_inspired(df):
    X = pd.DataFrame()
    X['PC_Feature1'] = df.GrLivArea + df.TotalBsmtSF
    X['PC_Feature2'] = df.YearRemodAdd * df.TotalBsmtSF
    return X


def label_encode(df):
    X = df.copy()
    for colname in X.select_dtypes(['category']).columns:
        X[colname] = X[colname].cat.codes
    return X


def create_features(df_train, df_test=None):
  # 1. Compute MI scores on training data ONLY
  y_train = df_train['SalePrice']
  X_train_raw = df_train.drop(columns=['SalePrice'], errors='ignore')

  mi_scores = make_mi_scores(X_train_raw, y_train.fillna(y_train.median()))
  keep_cols = mi_scores[mi_scores > 0.0].index.tolist()

  # 2. Combine train and test for consistent transformations
  if df_test is not None:
    X_test_raw = df_test.drop(columns=['SalePrice'], errors='ignore')
    df_combined = pd.concat([X_train_raw, X_test_raw], axis=0)
  else:
    df_combined = X_train_raw.copy()

  valid_cols = [c for c in keep_cols if c in df_combined.columns]
  X = df_combined[valid_cols].copy()

  # 3. Add engineered features
  X = X.join(mathematical_transforms(X))
  X = X.join(interactions(X))
  X = X.join(counts(X))
  X = X.join(group_transforms(X))
  X = X.join(pca_inspired(X))
  X = label_encode(X)

  # 4. Target Encoding (CV-based to prevent leakage)
  encoder = CrossFoldEncoder(MEstimateEncoder, m=1)
  encoded_train = encoder.fit_transform(
      X.loc[df_train.index], y_train, cols=['MSSubClass']
  )
  X.loc[df_train.index, encoded_train.columns] = encoded_train.values

  if df_test is not None:
    encoded_test = encoder.transform(X.loc[df_test.index])
    X.loc[df_test.index, encoded_test.columns] = encoded_test.values
    X_train_out = X.loc[df_train.index].copy()
    X_test_out = X.loc[df_test.index].copy()
    return X_train_out, X_test_out

  return X.loc[df_train.index].copy()

# Step 5: Run Baseline vs Engineered Model & Submit

In [6]:
if __name__ == '__main__':
    # Load dataset
    df_train, df_test = load_data()
    
    # 1. Baseline Score
    X_base = df_train.drop(columns=['SalePrice']).copy()
    y_base = df_train['SalePrice']
    for col in X_base.select_dtypes(['category']).columns:
        X_base[col] = X_base[col].cat.codes
    print(f"Baseline RMSLE Score: {score_dataset(X_base, y_base):.5f}")

    # 2. Engineered Features Score
    X_train_engineered = create_features(df_train)
    print(f"Engineered Features RMSLE Score: {score_dataset(X_train_engineered, y_base):.5f}")

    # 3. Tuned XGBoost Training & Final Submission
    X_train_feat, X_test_feat = create_features(df_train, df_test)
    y_train = df_train['SalePrice']

    xgb_params = dict(
        max_depth=5,
        learning_rate=0.03,
        n_estimators=1500,
        min_child_weight=2,
        colsample_bytree=0.6,
        subsample=0.8,
        reg_alpha=0.5,
        reg_lambda=1.5,
        random_state=42
    )

    model = XGBRegressor(**xgb_params)
    model.fit(X_train_feat, np.log(y_train))
    
    predictions = np.exp(model.predict(X_test_feat))
    
    submission = pd.DataFrame({'Id': X_test_feat.index, 'SalePrice': predictions})
    submission.to_csv('my_submission.csv', index=False)
    print("✅ Submission successfully generated as 'my_submission.csv'")

Baseline RMSLE Score: 0.14376
Engineered Features RMSLE Score: 0.14214
✅ Submission successfully generated as 'my_submission.csv'
